In [21]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# 1. Define Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# 2. Load Datasets
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
val_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

# 3. Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Verify the batch shape
dataiter = iter(train_loader)
images, labels = next(dataiter)
print("Batch image shape:", images.shape) # Output: torch.Size([32, 1, 28, 28])

Batch image shape: torch.Size([32, 1, 28, 28])


In [25]:
# Define the CNN architecture using PyTorch Sequential
cnn = nn.Sequential(
    # First Convolutional Block
    nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    # Second Convolutional Block
    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1),
    nn.ReLU(),
    nn.MaxPool2d(kernel_size=2, stride=2),

    # Flatten the 2D matrices into a 1D vector
    nn.Flatten(),

    # Dense/Linear Layers
    # 64 channels * 7 height * 7 width (since 28x28 is pooled twice)
    nn.Linear(64 * 7 * 7, 128),
    nn.ReLU(),
    nn.Linear(128, 10) # 10 output classes for digits 0-9
)

print(cnn)

Sequential(
  (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU()
  (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (4): ReLU()
  (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (6): Flatten(start_dim=1, end_dim=-1)
  (7): Linear(in_features=3136, out_features=128, bias=True)
  (8): ReLU()
  (9): Linear(in_features=128, out_features=10, bias=True)
)


In [26]:

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=0.001)

epochs = 5

print("Starting CNN training...")

for epoch in range(epochs):
    # --- TRAINING PHASE ---
    cnn.train()
    running_loss = 0.0
    correct_train = 0
    total_train = 0

    for x_train_batch, y_train_batch in train_loader:
        optimizer.zero_grad()

        outputs = cnn(x_train_batch)
        loss = criterion(outputs, y_train_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)
        total_train += y_train_batch.size(0)
        correct_train += (predicted == y_train_batch).sum().item()

    # --- VALIDATION PHASE ---
    cnn.eval() #
    correct_val = 0
    total_val = 0


    with torch.no_grad():
        for x_val_batch, y_val_batch in val_loader:
            outputs = cnn(x_val_batch)
            _, predicted = torch.max(outputs.data, 1)
            total_val += y_val_batch.size(0)
            correct_val += (predicted == y_val_batch).sum().item()

    train_acc = 100 * correct_train / total_train
    val_acc = 100 * correct_val / total_val
    print(f"Epoch [{epoch+1}/{epochs}] | Loss: {running_loss/len(train_loader):.4f} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")

Starting CNN training...
Epoch [1/5] | Loss: 0.1132 | Train Acc: 96.53% | Val Acc: 98.37%
Epoch [2/5] | Loss: 0.0412 | Train Acc: 98.72% | Val Acc: 99.06%
Epoch [3/5] | Loss: 0.0273 | Train Acc: 99.11% | Val Acc: 98.69%
Epoch [4/5] | Loss: 0.0193 | Train Acc: 99.37% | Val Acc: 99.21%
Epoch [5/5] | Loss: 0.0153 | Train Acc: 99.50% | Val Acc: 99.08%
